## Standard imports

In [ ]:
# If the bloomberg file path is not available in your JN, please run the following command in your JN terminal: 
# /home/.userconf/add_mount.py /home/tqdata/data/bloomberg

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

## Notebook configuration
- Please set the **<span style="color:red">config['enddate']</span>** to the most recent Friday. Do not change any other config parameters. 

- The environment might take around one minute to load
- The  **Operators** are functions that can be used to apply a series of transformations to a data variable or alpha.
> [Introduction to Operators](https://trexsim.com/trexsim/pysim/tutorial2022/operators.html)
> 
> Find all available  operators in the [operators list](https://www.trexsim.com/trexsim/pysim/comptutorial/operators.html)

In [2]:
import operators as op              # find all available operators, https://www.trexsim.com/trexsim/pysim/comptutorial/operators.html
import utilities as utils
import strategy_utilities as sutil
from strategy_research import load_env
from tqdm.notebook import tqdm

config = {
    'enddate': 20241206,            # Please setting the config['enddate'] to the most recent Friday.
    'region': 'USA',
    'universe': 'top3000',
    'delay': 0,
    'booksize':1e6,
    'refit_interval': 'OneQuarter',
    'filter_function_uuid': 'none',
    'fit_function_uuid': 'none',
    'pp_function_uuid': 'none',
    'tf_function_uuid': 'none',
    'postalpha': False,
    'compress': False,    
    'load_cache': False,
    'save_cache': False,
    'skip_post_strategy': True,
    'skip_test_cases': True,
    'strat_dir': './'
}

def tf_function(preA, data): return preA
function_dict = {'tf_function': tf_function}

env = load_env(config['region'])
data = sutil.create_strategy_data(function_dict, config, 'full', env, use_pre_compressed=True)
data = sutil.add_builtin_functions(data, function_dict, config, env)

dates = data['dates'].copy()
ret1 = data['ret1'].copy()
tickerNameMaps=utils.construct_tickerNameMap(data['load_simvar']('tickerNameMaps'),data['dates'])

is container True
Variable numdates is loaded from /home/newprod/production/data/USA/simvars/numdates/numdates.20241206.raw.mat
Variable numstocks is loaded from /home/newprod/production/data/USA/simvars/numstocks/numstocks.20241206.raw.mat
Variable dates is loaded from /home/newprod/production/data/USA/simvars/dates/dates.20241206.raw.mat
Variable dates is loaded from /home/newprod/production/data/USA/simvars/dates/dates.20241206.adj_amend.mat 4765
Variable numdates is loaded from /home/newprod/production/data/USA/simvars/numdates/numdates.20241206.adj_amend.mat 4765
Variable startdate is loaded from /home/newprod/production/data/USA/simvars/startdate/startdate.20241206.adj_amend.mat 4765
Variable enddate is loaded from /home/newprod/production/data/USA/simvars/enddate/enddate.20241206.adj_amend.mat 4765
Variable spread_timeweighted_allday is loaded from /home/newprod/production/data/USA/simvars/spread_timeweighted_allday/spread_timeweighted_allday.20241206.adj_amend.ci 4765
Variable 

## functions to work with intraday minute stock data

In [3]:
import glob
def load_intra_dfs(file,data_idxes,datatype):

    df_hhmm = pd.read_csv(file, sep='|', skiprows=0, usecols=data_idxes,dtype={'ticker':str}).set_index('ticker')

    #Please change the "usecols" above to get the required data minutely data variables:
    #print(['ticker', 'datetime', 'open', 'high', 'low', 'close', 'numevents','volume'])    
    # ticker     : 0
    # datetime   : 1
    # open       : 2
    # high       : 3
    # low        : 4
    # close      : 5
    # numevents  : 6 ## numevents is numtrades
    # volume     : 7
    
    #df_hhmm = df_hhmm.drop_duplicates('ticker', keep='first').set_index('ticker')
    hhmm = file.rsplit('.', 1)[1]
    df_hhmm.columns = [f"{datatype}_{i}_{hhmm}" for i in df_hhmm.columns]
    return df_hhmm


def get_intra_data(date, hhmm, data_idxes, datatype = 'price'): 
    
    if datatype == 'bid':
        filepaths = glob.glob(f'/home/tqdata/data/bloomberg/{date}/intra_price/bid.*')
    elif datatype == 'ask':
        filepaths = glob.glob(f'/home/tqdata/data/bloomberg/{date}/intra_price/ask.*')     
    else:
        filepaths = glob.glob(f'/home/tqdata/data/bloomberg/{date}/intra_price/px.*')
    
    valid_file_paths=[]
    for file in filepaths:
        if (int(file.split('.')[-1])<= hhmm)&(int(file.split('.')[-1])>= 931):
           valid_file_paths.append(file)     
            
    if len(valid_file_paths)==0: 
        print(None)
        return None
    
    
    df = pd.concat([load_intra_dfs(x,data_idxes,datatype) for x in valid_file_paths], axis=1)    
    df=df[sorted(df.columns)]

    return df

## Getting started with above function - Examples

In [5]:
dt = 20090107

df = get_intra_data(dt, 1500, [0,5,7], 'price') #5 for close and 7 for volume      
df_bid = get_intra_data(dt, 1500, [0,7], 'bid') #7 for volume
df_ask = get_intra_data(dt, 1500, [0,7], 'ask') #7 for volume

In [10]:
# final make one final dataframe to work with
df = df.merge(df_ask,left_on='ticker',right_on='ticker',how='inner')
df = df.merge(df_bid,left_on='ticker',right_on='ticker',how='inner')
df

,price_close_0931,price_close_0932,price_close_0933,price_close_0934,price_close_0935,price_close_0936,price_close_0937,price_close_0938,price_close_0939,price_close_0940,...,bid_volume_1451,bid_volume_1452,bid_volume_1453,bid_volume_1454,bid_volume_1455,bid_volume_1456,bid_volume_1457,bid_volume_1458,bid_volume_1459,bid_volume_1500
ticker,,,,,,,,,,,,,,,,,,,,,
BAC,14.02,14.00,13.97,14.00,14.00,14.055,14.00,14.00,14.000,13.99,...,60843,83775,107422,117234,86078,74937,78374,26009,32373,46913
JPM,29.17,29.04,29.03,29.09,29.16,29.250,29.24,29.09,29.070,29.12,...,11819,10591,10211,14318,10345,9446,5708,2581,7892,4220
XOM,79.32,79.29,79.51,79.43,79.54,79.560,79.67,79.48,79.672,79.62,...,2789,4086,2374,4212,3332,2417,1494,1832,5345,3444
WFC,27.00,26.93,26.98,26.78,26.96,27.000,26.96,26.93,26.890,26.93,...,25445,15745,23681,10703,10608,10958,7260,8642,21560,9286
PBR,26.78,26.77,26.82,26.86,26.87,26.890,26.95,26.81,26.800,26.76,...,4757,1495,3359,7368,8429,4485,15570,11548,25061,7340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NWLI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,0,0,2,4,0,0
UBSH,23.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,25,10,3,25,7,66,0,1,6,23
WM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


In [12]:
tickers = df.index
close = df[[col for col in df if col.startswith('price_close')]]
volume = df[[col for col in df if col.startswith('price_volume')]]
bid_volume = df[[col for col in df if col.startswith('bid_volume')]]
ask_volume = df[[col for col in df if col.startswith('ask_volume')]]


In [ ]:
bid_volume

In [ ]:
ask_volume

In [ ]:
final_dt = np.where(dates==20201231)[0][0]
new_dates = dates[:final_dt]
new_var = np.full((ret1.shape[0], ret1.shape[1]),np.nan,dtype=np.float32)

for di, dt in tqdm(enumerate(new_dates)):
    print(di, dt)
    try:
        ticker2si=pd.DataFrame({'ticker':tickerNameMaps[dt].keys(),'si':tickerNameMaps[dt].values()})
        
        ## do your changes here
        
        #Note: we only want to use the data upto 1500 (3pm only), not beyond that on any given day
        df = get_intra_data(dt, 1500, [0,5,7], 'price') #5 for close and 7 for volume      
        
        #only use merge when loading data from multiple files : price / bid / ask
        df_bid = get_intra_data(dt, 1500, [0,7], 'bid') #7 for volume
        df_ask = get_intra_data(dt, 1500, [0,7], 'ask') #7 for volume
        df = df.merge(df_ask,left_on='ticker',right_on='ticker',how='inner')
        df = df.merge(df_bid,left_on='ticker',right_on='ticker',how='inner')
        
        tickers = df.index
        close = df[[col for col in df if col.startswith('price_close')]].values
        volume = df[[col for col in df if col.startswith('price_volume')]].values
        bid_volume = df[[col for col in df if col.startswith('bid_volume')]].values
        ask_volume = df[[col for col in df if col.startswith('ask_volume')]].values

        # Two examples to use above data to create new data variables:
        # example-1 - calculation pv corr
        # y = (np.nanmean(close*volume, axis=1) - (np.nanmean(close, axis=1)*np.nanmean(volume, axis=1))) / (np.nanstd(close, axis=1)*np.nanstd(volume, axis=1))
        
        # example-2 - calculation bid ask volume spread
        y = np.nanstd((bid_volume - ask_volume) / (bid_volume + ask_volume), axis=1)
        
        dict = {'ticker':tickers, 'y':y}
        
        ## end of doing your changes
        
        df_y = pd.DataFrame(dict)
        df_y = df_y.merge(ticker2si,left_on='ticker',right_on='ticker',how='inner')
        
        new_var[df_y['si'].values,di] = df_y['y'].values

    except Exception as e:
        print(e)

In [28]:
close.shape

(2705, 330)

In [29]:
def safe_div(x, y):
        y = np.where(np.abs(y) < 1e-9, np.nan, y)
        return x / y

In [ ]:
final_dt = np.where(dates==20201231)[0][0]
new_dates = dates[:final_dt]
new_var = np.full((ret1.shape[0], ret1.shape[1]),np.nan,dtype=np.float32)

for di, dt in enumerate(tqdm(new_dates)):
    if dt<=20080107:
        continue
    print(di, dt)
    try:
        ticker2si=pd.DataFrame({'ticker':tickerNameMaps[dt].keys(),'si':tickerNameMaps[dt].values()})
        
        ## do your changes here
        
        #Note: we only want to use the data upto 1500 (3pm only), not beyond that on any given day
        df = get_intra_data(dt, 1500, [0,5], 'price') #5 for close and 7 for volume      
        
        #only use merge when loading data from multiple files : price / bid / ask
        # df_bid = get_intra_data(dt, 1500, [0,7], 'bid') #7 for volume
        # df_ask = get_intra_data(dt, 1500, [0,7], 'ask') #7 for volume
        # df = df.merge(df_ask,left_on='ticker',right_on='ticker',how='inner')
        # df = df.merge(df_bid,left_on='ticker',right_on='ticker',how='inner')
        
        tickers = df.index
        close = df[[col for col in df if col.startswith('price_close')]].values
        # volume = df[[col for col in df if col.startswith('price_volume')]].values
        # bid_volume = df[[col for col in df if col.startswith('bid_volume')]].values
        # ask_volume = df[[col for col in df if col.startswith('ask_volume')]].values
    
        # Two examples to use above data to create new data variables:
        # example-1 - calculation pv corr
        # y = (np.nanmean(close*volume, axis=1) - (np.nanmean(close, axis=1)*np.nanmean(volume, axis=1))) / (np.nanstd(close, axis=1)*np.nanstd(volume, axis=1))
    
        ret_min = np.log(safe_div(close, op.ts_delay(close, 1)))
        taylor = np.exp(ret_min) - 1 - ret_min - ret_min **2 /2
        taylor_daily = np.nanmean(taylor,axis=1)
    
    
        
        dict = {'ticker':tickers, 'y':taylor_daily}
        
        ## end of doing your changes
        
        df_y = pd.DataFrame(dict)
        df_y = df_y.merge(ticker2si,left_on='ticker',right_on='ticker',how='inner')
        
        new_var[df_y['si'].values,di] = df_y['y'].values
        
        # new_di_values = df_y['y'].values
        # print(f'di = {di}, date = {dt}, percentage of finite values avaiable : {len(new_di_values[~np.isnan(op.at_zero2nan(new_di_values))])*100 / (len(new_di_values)+1e-6)}%')

    

    except Exception as e:
        print(e)

  0%|          | 0/3775 [00:00<?, ?it/s]

506 20080108
507 20080109
508 20080110
509 20080111
510 20080114
511 20080115
512 20080116
513 20080117
514 20080118
515 20080122
516 20080123
517 20080124
518 20080125
519 20080128
520 20080129
521 20080130
522 20080131
523 20080201
524 20080204
525 20080205
526 20080206
527 20080207
528 20080208
529 20080211
530 20080212
531 20080213
532 20080214
533 20080215
534 20080219
535 20080220
536 20080221
537 20080222
538 20080225
539 20080226
540 20080227
541 20080228
542 20080229
543 20080303
544 20080304
545 20080305
546 20080306
547 20080307
548 20080310
549 20080311
550 20080312
551 20080313
552 20080314
553 20080317
554 20080318
555 20080319
556 20080320
557 20080324
558 20080325
559 20080326
560 20080327
561 20080328
562 20080331
563 20080401
564 20080402
565 20080403
566 20080404
567 20080407
568 20080408
569 20080409
570 20080410
571 20080411
572 20080414
573 20080415
574 20080416
575 20080417
576 20080418
577 20080421
578 20080422
579 20080423
580 20080424
581 20080425
582 20080428

In [40]:


gc.collect()

2048